# Data Cleaning - Railway Dataset

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

print("Libraries Imported Successfully")

Libraries Imported Successfully


## 2. Connect to Staging Database & Load Data

In [2]:

server = r".\SQLEXPRESS"
staging_database = "staging_railway"

staging_connection_string = (
    f"mssql+pyodbc://{server}/{staging_database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

staging_engine = create_engine(staging_connection_string)
df = pd.read_sql("SELECT * FROM railway", staging_engine)

print("Data loaded from staging successfully!")

Data loaded from staging successfully!


## 3. Data Cleaning & Transformation
### 3.1 Replacing Nulls

In [3]:
df['Railcard'] = df['Railcard'].fillna('No Railcard')
df['Reason for Delay'] = df['Reason for Delay'].fillna('No Delay')

### 3.2 Convert Date & Time Columns

In [4]:
# Convert Dates
df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'])
df['Date of Journey'] = pd.to_datetime(df['Date of Journey'])

# Convert Times to temporary datetime for calculation
df['Arrival_DT'] = pd.to_datetime(df['Arrival Time'], format="%H:%M:%S")
df['Actual_Arrival_DT'] = pd.to_datetime(df['Actual Arrival Time'], format="%H:%M:%S")

# Convert other times directly to time format
df['Time of Purchase'] = pd.to_datetime(df['Time of Purchase'], format="%H:%M:%S").dt.time
df['Departure Time'] = pd.to_datetime(df['Departure Time'], format="%H:%M:%S").dt.time

### 3.3 Calculate Delay Minutes (Handling Cross-Date Constraints)

In [5]:
# Calculate basic difference
df['Delay Time'] = df['Actual_Arrival_DT'] - df['Arrival_DT']
df['Delay Minutes'] = df['Delay Time'].dt.total_seconds() / 60

# --- THE FIX: Handle cross-date negative delays ---
df['Delay Minutes'] = np.where(df['Delay Minutes'] < 0, df['Delay Minutes'] + 1440, df['Delay Minutes'])

# --- THE FIX: Handle Cancelled trains (Null actual arrival time) ---
df['Delay Minutes'] = df['Delay Minutes'].fillna(0)

# Revert times back to pure time format and drop temporary columns
df['Arrival Time'] = df['Arrival_DT'].dt.time
df['Actual Arrival Time'] = df['Actual_Arrival_DT'].dt.time
df.drop(columns=['Arrival_DT', 'Actual_Arrival_DT', 'Delay Time'], inplace=True)

### 3.4 Remove Duplicates

In [6]:
df = df.drop_duplicates()
print(f"Final Cleaned Data Shape: {df.shape}")

Final Cleaned Data Shape: (31653, 19)


## 4. Send Cleaned Data To Clean Database

In [ ]:
clean_database = "cleaning_railway"
server = r".\SQLEXPRESS"

clean_connection_string = (
    f"mssql+pyodbc://{server}/{clean_database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)

clean_engine = create_engine(clean_connection_string)

df.to_sql(
    "clean_railway",
    clean_engine,
    if_exists="replace",
    index=False
)

print("Cleaning completed successfully and saved to 'clean_railway'!")

Cleaning completed successfully and saved to 'clean_railway'!
